# Baseline vs GraphSAINT 비교 분석

**실행 전 체크리스트**
- [ ] VS Code에서 Colab 런타임 연결 (Jupyter: Select Kernel → Existing Jupyter Server → Colab URL+token)
- [ ] Google Drive에 Baseline/GraphSAINT prediction CSV, meta JSON 저장 완료
- [ ] **경로 설정 셀** (`BASELINE_NAME`, `SAINT_NAME` 등) 수정

**분석 항목**
1. TP / FP / FN / TN 비교
2. Feature 수 vs 성능 비교
3. High-risk tail count (prob ≥ 0.7 / 0.8 / 0.9 / 0.95)
4. Cumulative Capture Curve (CCC)
5. Recall@K
6. Precision-Recall Curve
7. Score Distribution (fraud vs normal)
8. Amount-weighted Recall

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 경로 설정 ← 여기만 수정

In [ ]:
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/Graph_AML')
GNN_DIR    = DRIVE_BASE / 'gnn'
GNN_INPUTS = DRIVE_BASE / 'data' / 'processed' / 'gnn_inputs'
COMPARE_OUT = GNN_DIR / 'compare'
COMPARE_OUT.mkdir(parents=True, exist_ok=True)

# ─── Baseline ──────────────────────────────────────────────
BASELINE_EXPERIMENT = 'GNN-00'
BASELINE_RUN        = 'r01'
BASELINE_NAME       = 'small_hi_epoch100_patience20_numneighs100100'   # ← unique_name 수정

# ─── GraphSAINT ────────────────────────────────────────────
SAINT_EXPERIMENT    = 'GNN-04'
SAINT_RUN           = 'r07'
SAINT_NAME          = 'small_hi_graphsaint_rw_wl4_ns100_bs1000_lrs_nhidden64'             # ← unique_name 수정

BASELINE_RUN_DIR = GNN_DIR / BASELINE_EXPERIMENT / BASELINE_RUN
SAINT_RUN_DIR    = GNN_DIR / SAINT_EXPERIMENT    / SAINT_RUN

BASELINE_PRED_PATH   = BASELINE_RUN_DIR / f'{BASELINE_NAME}_predictions.csv'
SAINT_PRED_PATH      = SAINT_RUN_DIR    / f'{SAINT_NAME}_predictions.csv'
BASELINE_META_PATH   = BASELINE_RUN_DIR / f'{BASELINE_NAME}_meta.json'
SAINT_META_PATH      = SAINT_RUN_DIR    / f'{SAINT_NAME}_meta.json'
BASELINE_AMOUNT_PATH = BASELINE_RUN_DIR / f'{BASELINE_NAME}_predictions_with_amount.csv'
SAINT_AMOUNT_PATH    = SAINT_RUN_DIR    / f'{SAINT_NAME}_predictions_with_amount.csv'

for p in [BASELINE_PRED_PATH, SAINT_PRED_PATH, BASELINE_META_PATH, SAINT_META_PATH]:
    print(f"{'[OK]' if p.exists() else '[MISSING]'} {p}")
for p in [BASELINE_AMOUNT_PATH, SAINT_AMOUNT_PATH]:
    print(f"{'[OK]' if p.exists() else '[optional-missing]'} {p}")

## 데이터 로딩

In [ ]:
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# 폰트 파일 직접 등록 (캐시 재빌드 없이 즉시 적용)
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 적용 확인
test_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
print(f'등록된 Nanum 폰트: {test_fonts}')
print('한글 폰트 설정 완료')

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.ticker as mticker
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    f1_score, recall_score, precision_score,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

bl_pred = pd.read_csv(BASELINE_PRED_PATH)
gs_pred = pd.read_csv(SAINT_PRED_PATH)

with open(BASELINE_META_PATH) as f: bl_meta = json.load(f)
with open(SAINT_META_PATH)    as f: gs_meta = json.load(f)

# -1 (미수집 엣지) 제거
bl_valid = bl_pred[bl_pred['gt'] >= 0].copy().reset_index(drop=True)
gs_valid = gs_pred[gs_pred['gt'] >= 0].copy().reset_index(drop=True)

bl_amount = pd.read_csv(BASELINE_AMOUNT_PATH) if BASELINE_AMOUNT_PATH.exists() else None
gs_amount = pd.read_csv(SAINT_AMOUNT_PATH)    if SAINT_AMOUNT_PATH.exists()    else None
if bl_amount is not None: bl_amount = bl_amount[bl_amount['gt'] >= 0].copy()
if gs_amount is not None: gs_amount = gs_amount[gs_amount['gt'] >= 0].copy()

MODELS = {
    'Baseline':   {'pred': bl_valid, 'meta': bl_meta, 'amount': bl_amount, 'color': '#1565C0'},
    'GraphSAINT': {'pred': gs_valid, 'meta': gs_meta, 'amount': gs_amount, 'color': '#BF360C'},
}

print(f'Baseline   valid: {len(bl_valid):,} rows  (fraud={int(bl_valid["gt"].sum()):,})')
print(f'GraphSAINT valid: {len(gs_valid):,} rows  (fraud={int(gs_valid["gt"].sum()):,})')

## 분석 1 — TP / FP / FN / TN 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
group_colors = {'TP': '#388E3C', 'FP': '#F57C00', 'FN': '#C62828', 'TN': '#757575'}

for ax, (name, info) in zip(axes, MODELS.items()):
    m     = info['meta']['test_metrics']
    keys  = ['TP', 'FP', 'FN', 'TN']
    vals  = [m['tp'], m['fp'], m['fn'], m['tn']]
    clrs  = [group_colors[k] for k in keys]
    bars  = ax.bar(keys, vals, color=clrs, edgecolor='white', linewidth=1.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.01,
                f'{v:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    title_line2 = (f'F1={m["f1"]:.4f}  Recall={m["recall"]:.4f}\n'
                   f'Precision={m["precision"]:.4f}  AUPRC={m["auprc"]:.4f}')
    ax.set_title(f'{name}\n{title_line2}', fontsize=11)
    ax.set_ylabel('Count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('TP / FP / FN / TN 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(COMPARE_OUT / 'a1_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {COMPARE_OUT / "a1_confusion.png"}')

## 분석 2 — Feature 수 vs 성능 비교

In [ ]:
def count_features(meta):
    return (meta.get('n_base_edge_features', 0)
            + meta.get('n_port_features', 0)
            + meta.get('n_tds_features', 0)
            + meta.get('n_ego_node_features', 0))

rows = []
for name, info in MODELS.items():
    m = info['meta']
    tm = m['test_metrics']
    rows.append({
        'Model':         name,
        'Base features': m.get('n_base_edge_features', 0),
        'Port features': m.get('n_port_features', 0),
        'TDS features':  m.get('n_tds_features', 0),
        'Ego features':  m.get('n_ego_node_features', 0),
        'Total':         count_features(m),
        'F1':            tm['f1'],
        'Recall':        tm['recall'],
        'Precision':     tm['precision'],
        'AUPRC':         tm['auprc'],
    })
feat_df = pd.DataFrame(rows).set_index('Model')

print(feat_df.to_string())

# Grouped bar: 4 metrics side by side
metrics   = ['F1', 'Recall', 'Precision', 'AUPRC']
names     = list(MODELS.keys())
colors    = [info['color'] for info in MODELS.values()]
x         = np.arange(len(metrics))
w         = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
for i, (name, clr) in enumerate(zip(names, colors)):
    vals = [feat_df.loc[name, m] for m in metrics]
    bars = ax.bar(x + i * w, vals, w, label=f'{name} (feat={int(feat_df.loc[name,"Total"])})',
                  color=clr, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.003,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + w / 2)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Feature 수 vs 성능 비교')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(COMPARE_OUT / 'a2_feature_perf.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {COMPARE_OUT / "a2_feature_perf.png"}')

## 분석 3 — High-risk Tail Count (prob ≥ threshold)

In [ ]:
from scipy.stats import gaussian_kde

TAIL_MIN = 0.90  # x축 시작점

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, info) in zip(axes, MODELS.items()):
    pred = info['pred']
    thr  = info['meta'].get('threshold', 0.5)

    x_grid = np.linspace(TAIL_MIN, 1.0, 400)

    for gt_val, clr, lbl in [(0, '#5B9BD5', 'normal'), (1, '#E07070', 'laundering')]:
        vals      = pred[pred['gt'] == gt_val]['prob'].values
        tail_vals = vals[vals >= TAIL_MIN]
        if len(tail_vals) < 2:
            continue
        kde     = gaussian_kde(tail_vals, bw_method='silverman')
        density = kde(x_grid)
        ax.fill_between(x_grid, density, alpha=0.45, color=clr, label=lbl)
        ax.plot(x_grid, density, color=clr, linewidth=1.5)

    # xlim 먼저 설정 — 이후 추가되는 요소가 범위를 벗어나도 영향 없음
    ax.set_xlim(TAIL_MIN, 1.0)

    # threshold 수직선: 가시 범위 안에 있을 때만 표시
    if TAIL_MIN <= thr <= 1.0:
        ax.axvline(thr, color='black', linestyle='--', linewidth=1.5, alpha=0.85)
        ymax = ax.get_ylim()[1]
        ax.text(thr + 0.002, ymax * 0.82, f'threshold\n{thr:.4f}',
                fontsize=8, va='top', color='black',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none'))

    ax.set_xlabel('Predicted score')
    ax.set_ylabel('Density')
    ax.set_title(f'{name} 고위험 tail 분포')
    ax.legend(title='class')
    ax.grid(alpha=0.3)

plt.suptitle('High-Risk Tail Distribution on Test Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(COMPARE_OUT / 'a3_high_risk_tail.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {COMPARE_OUT / "a3_high_risk_tail.png"}')

## 분석 4 — Cumulative Capture Curve (CCC)

x축: 전체 거래 중 상위 k% 검토 / y축: illicit 거래 중 몇 % 포착

In [ ]:
def compute_ccc(pred_df):
    df          = pred_df.sort_values('prob', ascending=False).reset_index(drop=True)
    total_fraud = int(df['gt'].sum())
    cum_fraud   = df['gt'].cumsum().values
    k_vals      = np.arange(1, len(df) + 1)          # Top-K (절대 건수)
    pct_captured = cum_fraud / total_fraud * 100      # y축: %
    return k_vals, pct_captured

fig, ax = plt.subplots(figsize=(9, 6))
for name, info in MODELS.items():
    k_vals, y_ccc = compute_ccc(info['pred'])
    ax.plot(k_vals, y_ccc, label=name, color=info['color'], linewidth=2)

ax.set_xscale('log')
ax.set_xlabel('Top-K transactions by predicted score, log scale')
ax.set_ylabel('Captured laundering transactions (%)')
ax.set_ylim(0, 100)
ax.set_title('Cumulative Laundering Capture by Investigation Budget')
ax.legend(title='Model')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(COMPARE_OUT / 'a4_ccc.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {COMPARE_OUT / "a4_ccc.png"}')

## 분석 5 — Recall@K

상위 K% 거래를 플래그했을 때의 recall

In [ ]:
def compute_recall_at_k(pred_df):
    df          = pred_df.sort_values('prob', ascending=False).reset_index(drop=True)
    total_fraud = int(df['gt'].sum())
    k_vals      = np.arange(1, len(df) + 1)
    recalls     = df['gt'].cumsum().values / total_fraud
    return k_vals, recalls

fig, ax = plt.subplots(figsize=(9, 6))

k_arrays, recall_arrays = [], []
for name, info in MODELS.items():
    k_vals, recalls = compute_recall_at_k(info['pred'])
    ax.plot(k_vals, recalls, marker='o', markersize=0,
            label=name, color=info['color'], linewidth=2)
    k_arrays.append(k_vals)
    recall_arrays.append(recalls)

# 두 모델 간 최대 gap 표시
if len(recall_arrays) == 2:
    # 공통 K 범위에서 gap 계산
    min_len = min(len(k_arrays[0]), len(k_arrays[1]))
    gap     = np.abs(recall_arrays[0][:min_len] - recall_arrays[1][:min_len])
    max_idx = int(np.argmax(gap))
    max_k   = int(k_arrays[0][max_idx])
    max_gap = float(gap[max_idx])
    r0      = float(recall_arrays[0][max_idx])
    r1      = float(recall_arrays[1][max_idx])
    r_top, r_bot = max(r0, r1), min(r0, r1)

    ax.annotate('', xy=(max_k, r_bot), xytext=(max_k, r_top),
                arrowprops=dict(arrowstyle='<->', color='black', lw=1.5, linestyle='dashed'))
    ax.scatter([max_k, max_k], [r_top, r_bot], s=60, zorder=5,
               color=[list(MODELS.values())[0]['color'], list(MODELS.values())[1]['color']])
    ax.annotate(f'최대 gap +{max_gap:.3f}\nK={max_k:,}',
                xy=(max_k, (r_top + r_bot) / 2),
                xytext=(max_k * 1.3, (r_top + r_bot) / 2),
                fontsize=8, va='center',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.8),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

ax.set_xscale('log')
ax.set_xlabel('Top-K transactions by predicted score, log scale')
ax.set_ylabel('Recall@K')
ax.set_ylim(0, 1.0)
ax.set_title('Recall@K on Test Score Ranking')
ax.legend(title='Model')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(COMPARE_OUT / 'a5_recall_at_k.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {COMPARE_OUT / "a5_recall_at_k.png"}')

# 주요 K값 요약 테이블
summary_rows = []
for name, info in MODELS.items():
    df = info['pred'].sort_values('prob', ascending=False).reset_index(drop=True)
    total_fraud = int(df['gt'].sum())
    row = {'Model': name}
    for k in [100, 500, 1000, 5000, 10000]:
        if k <= len(df):
            row[f'Recall@{k:,}'] = round(int(df.iloc[:k]['gt'].sum()) / total_fraud, 4)
    summary_rows.append(row)
print(pd.DataFrame(summary_rows).set_index('Model').to_string())

## 분석 6 — Precision-Recall Curve

ROC보다 class imbalance에서 더 정직한 지표

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, info in MODELS.items():
    pred  = info['pred']
    prec, rec, _ = precision_recall_curve(pred['gt'], pred['prob'])
    auprc = average_precision_score(pred['gt'], pred['prob'])
    ax.plot(rec, prec, label=f'{name} (AUPRC={auprc:.4f})',
            color=info['color'], linewidth=2)

# 랜덤 베이스라인: illicit 비율
ir = float(bl_valid['gt'].mean())
ax.axhline(ir, color='gray', linestyle='--', linewidth=1.2, alpha=0.6,
           label=f'Random (IR={ir:.4f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.set_title('Precision-Recall Curve')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(COMPARE_OUT / 'a6_pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {COMPARE_OUT / "a6_pr_curve.png"}')

## 분석 7 — Score Distribution (fraud vs normal)

모델이 fraud/normal을 얼마나 잘 분리하는지 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
bins = np.linspace(0, 1, 51)

for col, (name, info) in enumerate(MODELS.items()):
    pred    = info['pred']
    illicit = pred[pred['gt'] == 1]['prob'].values
    normal  = pred[pred['gt'] == 0]['prob'].values

    # 히스토그램
    ax = axes[0][col]
    ax.hist(normal,  bins=bins, alpha=0.55, color='#1565C0', label='Normal',  density=True)
    ax.hist(illicit, bins=bins, alpha=0.65, color='#C62828', label='Illicit', density=True)
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2, alpha=0.6, label='thr=0.5')
    ax.set_title(f'{name} — Histogram')
    ax.set_xlabel('Fraud Probability')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    # 누적 분포 (CDF) — matplotlib 3.7 호환
    ax = axes[1][col]
    for arr, clr, lbl in [(normal, '#1565C0', 'Normal'), (illicit, '#C62828', 'Illicit')]:
        s = np.sort(arr)
        y = np.arange(1, len(s) + 1) / len(s)
        ax.step(s, y, color=clr, label=lbl, linewidth=2, where='post')
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2, alpha=0.6)
    ax.set_title(f'{name} — CDF')
    ax.set_xlabel('Fraud Probability')
    ax.set_ylabel('Cumulative Fraction')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Score Distribution — Normal vs Illicit', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(COMPARE_OUT / 'a7_score_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {COMPARE_OUT / "a7_score_dist.png"}')

## 분석 8 — Amount-weighted Recall

건수 기준 재현율이 아닌 **금액 기준** 재현율.
`7-4번 셀` 실행 결과인 `_predictions_with_amount.csv` 필요.
파일이 없으면 이 셀은 건너뜁니다.

In [ ]:
has_amount = bl_amount is not None or gs_amount is not None

if not has_amount:
    print('amount 데이터 없음 — 각 모델 노트북의 7-4번 셀 실행 후 다시 로딩하세요.')
else:
    K_FRACS_AMT = np.linspace(0.005, 0.5, 300)

    def amount_capture_curve(amount_df, k_fracs):
        if amount_df is None:
            return None
        df = amount_df.dropna(subset=['amount_usd']).copy()
        df = df.sort_values('prob', ascending=False).reset_index(drop=True)
        total_illicit_usd = df[df['gt'] == 1]['amount_usd'].sum()
        if total_illicit_usd <= 0:
            return None
        n = len(df)
        captures = []
        for frac in k_fracs:
            k = max(1, int(n * frac))
            top_k = df.iloc[:k]
            detected_usd = top_k[top_k['gt'] == 1]['amount_usd'].sum()
            captures.append(detected_usd / total_illicit_usd)
        return captures

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 왼쪽: Amount-weighted CCC
    ax = axes[0]
    for name, info in MODELS.items():
        caps = amount_capture_curve(info['amount'], K_FRACS_AMT)
        if caps is not None:
            ax.plot(K_FRACS_AMT * 100, caps, label=name, color=info['color'], linewidth=2)
    ax.plot([0, 50], [0, 1], 'k--', alpha=0.4, linewidth=1.2, label='Random')
    ax.set_xlabel('Top-K Transactions Flagged (%)')
    ax.set_ylabel('Illicit Amount Captured (fraction)')
    ax.set_title('Amount-weighted Capture Curve')
    ax.legend()
    ax.grid(alpha=0.3)

    # 오른쪽: Bar — amount-weighted recall at fixed thresholds
    ax = axes[1]
    THRS = [0.3, 0.5, 0.7, 0.9]
    x   = np.arange(len(THRS))
    w   = 0.35
    for i, (name, info) in enumerate(MODELS.items()):
        if info['amount'] is None:
            continue
        df = info['amount'].dropna(subset=['amount_usd'])
        total_illicit_usd = df[df['gt'] == 1]['amount_usd'].sum()
        aw_recalls = []
        for thr in THRS:
            detected = df[(df['gt'] == 1) & (df['prob'] >= thr)]['amount_usd'].sum()
            aw_recalls.append(detected / total_illicit_usd if total_illicit_usd > 0 else 0)
        bars = ax.bar(x + i * w, aw_recalls, w, label=name, color=info['color'], alpha=0.85)
        for bar, v in zip(bars, aw_recalls):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x + w / 2)
    ax.set_xticklabels([f'thr={t}' for t in THRS])
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Amount-weighted Recall')
    ax.set_title('Amount-weighted Recall by Threshold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Amount-weighted Recall 분석 (USD 기준)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(COMPARE_OUT / 'a8_amount_recall.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'저장: {COMPARE_OUT / "a8_amount_recall.png"}')

    # 요약 출력
    print('\n[Amount-weighted Recall @ thr=0.5]')
    for name, info in MODELS.items():
        if info['amount'] is None:
            print(f'  {name}: 데이터 없음')
            continue
        df = info['amount'].dropna(subset=['amount_usd'])
        total = df[df['gt'] == 1]['amount_usd'].sum()
        detected = df[(df['gt'] == 1) & (df['prob'] >= 0.5)]['amount_usd'].sum()
        print(f'  {name}: {detected/total:.4f}  (${detected:,.0f} / ${total:,.0f} USD)')

## 전체 요약 — 지표 비교표

In [ ]:
summary_data = []
for name, info in MODELS.items():
    m  = info['meta']['test_metrics']
    df = info['pred']
    total_fraud = int(df['gt'].sum())
    n = len(df)

    # recall@5%
    df_sorted = df.sort_values('prob', ascending=False).reset_index(drop=True)
    k5 = max(1, int(n * 0.05))
    r_at_5 = int(df_sorted.iloc[:k5]['gt'].sum()) / total_fraud

    summary_data.append({
        'Model':           name,
        'F1':              m['f1'],
        'Recall':          m['recall'],
        'Precision':       m['precision'],
        'AUPRC':           m['auprc'],
        'TP':              m['tp'],
        'FP':              m['fp'],
        'FN':              m['fn'],
        'TN':              m['tn'],
        'High-risk(>=0.9)':info['meta'].get('high_risk_count', int((df['prob'] >= 0.9).sum())),
        'Recall@5%':       round(r_at_5, 4),
    })

summary_df = pd.DataFrame(summary_data).set_index('Model')
pd.set_option('display.float_format', '{:.4f}'.format)
print(summary_df.to_string())

SUMMARY_CSV = COMPARE_OUT / 'summary.csv'
summary_df.to_csv(SUMMARY_CSV)
print(f'\n저장: {SUMMARY_CSV}')